In [2]:
# Run this in 01_data_exploration.ipynb — Cell 1
import sys, shutil, os
sys.path.append("..")

# Clean old indexes
for path in [
    "../data/processed/chunks.json",
    "../data/processed/manifest.json",
    "../vector_rag/chroma_db",
    "../vectorless_rag/bm25_index.pkl",
    "../vectorless_rag/bm25_manifest.json"
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"🗑️  {path}")
    elif os.path.exists(path):
        os.remove(path)
        print(f"🗑️  {path}")

print("\n✅ Clean — ready to rebuild")

🗑️  ../data/processed/chunks.json
🗑️  ../data/processed/manifest.json
🗑️  ../vector_rag/chroma_db

✅ Clean — ready to rebuild


In [3]:
# Cell 2 — Rebuild with new architecture
from data_loader             import run_preprocessing_pipeline
from vector_rag.indexer      import index_chunks
from vectorless_rag.indexer  import build_bm25_index

data = run_preprocessing_pipeline()

print(f"Parents  : {len(data['parents'])}")
print(f"Children : {len(data['children'])}")

index_chunks(data)
build_bm25_index(data)

print("\n🎉 All indexes rebuilt with improved architecture!")

   PHASE 2 — SCALABLE PREPROCESSING PIPELINE
  🆕 New PDF detected: amazon_10k.pdf
  🆕 New PDF detected: microsoft_10k.pdf
  🆕 New PDF detected: netflix_10k.pdf
  🆕 New PDF detected: nvidia_10k.pdf

📂 Processing 4 new PDF(s)...



Processing PDFs:  25%|██▌       | 1/4 [00:00<00:01,  1.99it/s]

   ✅ AMAZON_10K: 90 pages → 410 parents, 1468 children


Processing PDFs:  50%|█████     | 2/4 [00:03<00:04,  2.08s/it]

   ✅ MICROSOFT_10K: 156 pages → 629 parents, 2242 children


...

   ✅ NVIDIA_10K: 93 pages → 435 parents, 1914 children

💾 Saved 2010 parents, 7913 children
📋 Manifest: 4 PDFs tracked

🎉 Preprocessing complete!

Parents  : 2010
Children : 7913
   VECTOR RAG INDEXER — Parent-Child + BGE + HNSW


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📤 Indexing 7913 child chunks...
   Embedding model: BAAI/bge-base-en-v1.5


Embedding children: 100%|██████████| 124/124 [09:47<00:00,  4.74s/it]


✅ ChromaDB updated — 2289 total vectors

   VECTORLESS RAG INDEXER — BM25 + Financial Tokenizer

✅ BM25 already up to date — 7913 children indexed


🎉 All indexes rebuilt with improved architecture!


In [4]:
# Cell 3 — Quick retrieval test
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

q = "What was NVIDIA's total revenue?"

r1 = vec.ask(q)
r2 = vl.ask(q)

vec.show(r1)
vl.show(r2)

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 2289 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

🔁 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready

  METHOD  : Vector RAG — BGE + HNSW + Parent-Child
  Q: What was NVIDIA's total revenue?
───────────────────────────────────────────────────────
  A: For **NVIDIA Corporation**, the total revenue was:
- **$215.94 billion** for the fiscal year ended **January 25, 2026**
- **$130.497 billion** for the fiscal year ended **January 26, 2025**
- **$60.922 billion** for the fiscal year ended **January 28, 2024**.
───────────────────────────────────────────────────────
   • NVIDIA | Page 69 | Score 7.3471
   • NVIDIA | Page 51 | Score 6.6990
   • NVIDIA | Page 79 | Score 5.4078
   • NVIDIA | Page 55 | Score 4.0247
   • NVIDIA | Page 52 | Score 3.9045
───────────────────────────────────────────────────────
  Retrieval: 7.9096s | Generation: 3.2302s | Total: 11.1398s


  METHOD  : Vectorless RAG — BM25 + Financial Tokenizer
  Q: What was NVIDIA's total revenue?
───────────────────────────────────────────────────────
  A: For **NVIDIA**, the total revenue for the fiscal year end

In [5]:
import traceback
from vectorless_rag.pipeline import VectorlessRAGPipeline

try:
    vl = VectorlessRAGPipeline()
    r2 = vl.ask("What was NVIDIA's total revenue?")
    print("Answer:", r2)
    vl.show(r2)
except Exception as e:
    print("FULL ERROR:")
    traceback.print_exc()
    print("\nException details:", str(e))

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

Answer: {'question': "What was NVIDIA's total revenue?", 'answer': 'For **NVIDIA**, the total revenue for the fiscal year ended **January 25, 2026**, was **$139,297 million** (as stated in Source 3). The total revenue for the fiscal year ended **January 26, 2025**, was **$87,960 million**. Figures for other years were not explicitly provided in the context.', 'retrieved': [{'text': 'of total revenue, all of which were primarily attributable to the Compute & Networking segment.\nFor fiscal year 2025, sales to one direct customer represented 12% of total revenue and sales to two direct customers each represented 11% of total revenue, all\nof which were primarily attributable to the Compute & Networking segment.\nFor fiscal year 2024, sales to one direct customer represented 13% of total revenue, and were primarily attri